In [ ]:
from pyspark import SparkContext, SparkConf
import os

os.environ['PYSPARK_PYTHON'] = '/usr/bin/python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/usr/bin/python3'

conf = SparkConf().setAppName("AirportsAnalysis").setMaster("local[4]")
sc = SparkContext(conf=conf)

airports_rdd = sc.textFile("airports.csv")

header = airports_rdd.first()
airports_data = airports_rdd.filter(lambda line: line != header).filter(lambda line: len(line) > 0)

def parse_airport_line(line):
    fields = line.replace('"', '').split(',')
    if len(fields) >= 7:
        faa_code = fields[0].strip()
        airport_name = fields[1].strip()
        state = "Unknown"
        if ',' in airport_name:
            state = airport_name.split(',')[-1].strip()
        timezone = fields[5].strip() if fields[5] else "Unknown"
        return (faa_code, airport_name, state, timezone)
    return None

airports_tuples = airports_data.map(parse_airport_line).filter(lambda x: x is not None)

print("=== First 10 Airports ===")
first_10_airports = airports_tuples.take(10)
for airport in first_10_airports:
    print(airport)

print("\n" + "="*50 + "\n")

unique_states = airports_tuples.map(lambda x: x[2]).distinct()
print("=== Unique States ===")
print(unique_states.collect())

print("\n" + "="*50 + "\n")

airports_per_state = airports_tuples.map(lambda x: (x[2], 1)).reduceByKey(lambda a, b: a + b)
print("=== Number of Airports per State ===")
airports_count = airports_per_state.collect()
for state, count in sorted(airports_count, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{state}: {count} airports")

print("\n" + "="*50 + "\n")

airports_by_timezone = airports_tuples.map(lambda x: (x[3], x[1])).groupByKey()
print("=== Airports by Timezone ===")
timezone_airports = airports_by_timezone.mapValues(list).collect()
for timezone, airports in sorted(timezone_airports)[:5]:  
    print(f"Timezone {timezone}: {len(airports)} airports")
    for airport in airports[:3]:  
        print(f"  - {airport}")

print("\n" + "="*50 + "\n")


def parse_airport_with_altitude(line):
    
    fields = line.replace('"', '').split(',')
    if len(fields) >= 7:
        faa_code = fields[0].strip()
        airport_name = fields[1].strip()
        state = "Unknown"
        if ',' in airport_name:
            state = airport_name.split(',')[-1].strip()
        try:
            altitude = float(fields[4]) if fields[4] else 0.0
        except:
            altitude = 0.0
        return (state, altitude)
    return None

airports_with_altitude = airports_data.map(parse_airport_with_altitude).filter(lambda x: x is not None)

altitude_by_state = airports_with_altitude.map(lambda x: (x[0], (x[1], 1)))
altitude_stats = altitude_by_state.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
average_altitude = altitude_stats.mapValues(lambda x: round(x[0] / x[1], 2))

print("=== Average Altitude by State (Top 10) ===")
avg_altitude_results = average_altitude.collect()
for state, avg_alt in sorted(avg_altitude_results, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{state}: {avg_alt} feet")

print("\n" + "="*50 + "\n")

max_altitude_by_state = airports_with_altitude.reduceByKey(max)
print("=== Maximum Altitude by State (Top 10) ===")
max_altitude_results = max_altitude_by_state.collect()
for state, max_alt in sorted(max_altitude_results, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{state}: {max_alt} feet")

print("\n" + "="*50 + "\n")

highest_avg_state = average_altitude.reduce(lambda a, b: a if a[1] > b[1] else b)
print("=== State with Highest Average Airport Altitude ===")
print(f"{highest_avg_state[0]}: {highest_avg_state[1]} feet")

print("\n" + "="*50 + "\n")

timezone_distribution = airports_tuples.map(lambda x: (x[3], 1)).reduceByKey(lambda a, b: a + b)
print("=== Airport Distribution by Timezone ===")
timezone_results = timezone_distribution.collect()
for tz, count in sorted(timezone_results, key=lambda x: x[1], reverse=True):
    print(f"Timezone {tz}: {count} airports")

sc.stop()

=== First 10 Airports ===
('04G', 'Lansdowne Airport', 'Unknown', '-5')
('06A', 'Moton Field Municipal Airport', 'Unknown', '-5')
('06C', 'Schaumburg Regional', 'Unknown', '-6')
('06N', 'Randall Airport', 'Unknown', '-5')
('09J', 'Jekyll Island Airport', 'Unknown', '-4')
('0A9', 'Elizabethton Municipal Airport', 'Unknown', '-4')
('0G6', 'Williams County Airport', 'Unknown', '-5')
('0G7', 'Finger Lakes Regional Airport', 'Unknown', '-5')
('0P2', 'Shoestring Aviation Airfield', 'Unknown', '-5')
('0S9', 'Jefferson County Intl', 'Unknown', '-8')


=== Unique States ===
['Unknown']


=== Number of Airports per State ===
Unknown: 1397 airports


=== Airports by Timezone ===
Timezone -10: 26 airports
  - Atmautluak Airport
  - Adak Airport
  - Barking Sands Pmrf
Timezone -11: 2 airports
  - St George
  - St Paul Island
Timezone -4: 66 airports
  - Jekyll Island Airport
  - Elizabethton Municipal Airport
  - Jackson County Airport
Timezone -5: 450 airports
  - Lansdowne Airport
  - Moton Field